In [9]:

import spark_patch
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, avg, length, count
import pandas as pd

spark = SparkSession.builder.appName("customerVisitedButNoTransactions").getOrCreate()

data = [[1, 23], [2, 9], [4, 30], [5, 54], [6, 96], [7, 54], [8, 54]]
visits = pd.DataFrame(data, columns=['visit_id', 'customer_id']).astype({'visit_id':'Int64', 'customer_id':'Int64'})
data = [[2, 5, 310], [3, 5, 300], [9, 5, 200], [12, 1, 910], [13, 2, 970]]
transactions = pd.DataFrame(data, columns=['transaction_id', 'visit_id', 'amount']).astype({'transaction_id':'Int64', 'visit_id':'Int64', 'amount':'Int64'})

visits_df = spark.createDataFrame(visits)
transactions_df = spark.createDataFrame(transactions)

visits_df.join(transactions_df, on='visit_id', how='left_anti').select('customer_id')\
    .groupBy('customer_id')\
    .agg(count('customer_id').alias('no_transaction_count'))\
    .show()

spark.stop()



+-----------+--------------------+
|customer_id|no_transaction_count|
+-----------+--------------------+
|         54|                   2|
|         96|                   1|
|         30|                   1|
+-----------+--------------------+

